# Phase 1 — Behavioral Segmentation

Segments customers into income behavior groups before income estimation.

## Segments
| Label | Description |
|---|---|
| PAYROLL | SCB payroll credit — income known |
| SALARY_LIKE | Regular monthly credits, low variance |
| SME | Business cash flow patterns |
| GIG_FREELANCE | Irregular credits, multiple sources |
| PASSIVE_INVESTOR | Interest/dividend income |
| THIN | Insufficient data |

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

from src.segmentation import SegmentationPipeline
from src.utils import setup_logging

setup_logging('INFO')

## Load Features

In [ ]:
# Load monthly aggregate features
# features_df = pd.read_parquet('../data/processed/monthly_features.parquet')

# --- Sample data for development ---
from data.sample.generate_sample import generate_sample_features
features_df = generate_sample_features(n=10_000)
print(features_df.shape)
features_df.head()

## Run Segmentation Pipeline

In [ ]:
pipeline = SegmentationPipeline(config_path='../config/config.yaml')
segmented_df = pipeline.run(features_df)
pipeline.get_summary(segmented_df)

## Segment Profiling

In [ ]:
profile_cols = [
    'avg_monthly_credit_12m', 'cv_monthly_credit_12m',
    'avg_eom_balance_3m', 'months_data_available',
    'avg_commitment_amount_12m'
]
segmented_df.groupby('segment')[profile_cols].median().round(0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

segmented_df['segment'].value_counts().plot(kind='bar', ax=axes[0], title='Segment Distribution')

sns.boxplot(
    data=segmented_df[segmented_df['avg_monthly_credit_12m'] < 200_000],
    x='segment', y='avg_monthly_credit_12m', ax=axes[1]
)
axes[1].set_title('Monthly Credit by Segment')
plt.tight_layout()
plt.show()